# Stop Dashboard Animation

In [1]:
%matplotlib inline

from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import HTML

import nomad.data as data_folder
import nomad.filters as filters
import nomad.io.base as loader
import nomad.stop_detection.dbscan as DBSCAN
import nomad.stop_detection.density_based as SEQSCAN
import nomad.stop_detection.grid_based as GRID_BASED
import nomad.stop_detection.hdbscan as HDBSCAN
import nomad.stop_detection.lachesis as LACHESIS
from nomad.stop_detection.viz import animate_stop_dashboard

In [2]:
tc = {
    "user_id": "gc_identifier",
    "x": "dev_x",
    "y": "dev_y",
    "timestamp": "unix_ts",
}

data_dir = Path(data_folder.__file__).parent
repo_root = Path(data_folder.__file__).resolve().parents[2]

traj = loader.sample_from_file(
    repo_root / "examples" / "gc_data_long",
    format="parquet",
    users=["admiring_brattain"],
    filters=("date", "==", "2024-01-01"),
    traj_cols=tc,
)
traj["h3_cell"] = filters.to_tessellation(
    traj,
    index="h3",
    res=11,
    traj_cols=tc,
    data_crs="EPSG:3857",
)
city = gpd.read_parquet(data_dir / "garden-city-buildings-mercator.parquet")

In [3]:
animation_cases = {
    "lachesis": {
        "run": LACHESIS.lachesis,
        "labels": LACHESIS.lachesis_labels,
        "params": {"delta_roam": 20, "dt_max": 60, "dur_min": 5},
    },
    "seqscan": {
        "run": SEQSCAN.seqscan,
        "labels": SEQSCAN.seqscan_labels,
        "params": {"time_thresh": 60, "dist_thresh": 8, "min_pts": 3},
    },
    "hdbscan": {
        "run": HDBSCAN.st_hdbscan,
        "labels": HDBSCAN.hdbscan_labels,
        "params": {"time_thresh": 720, "min_pts": 3},
    },
    "ta_dbscan": {
        "run": DBSCAN.ta_dbscan,
        "labels": DBSCAN.ta_dbscan_labels,
        "params": {"time_thresh": 60, "dist_thresh": 10, "min_pts": 3},
    },
    "grid_based": {
        "run": GRID_BASED.grid_based,
        "labels": GRID_BASED.grid_based_labels,
        "params": {"time_thresh": 240, "location_id": "h3_cell"},
    },
}

In [4]:
case = animation_cases["lachesis"]
stops = case["run"](
    traj,
    complete_output=True,
    traj_cols=tc,
    **case["params"],
)

fig, (ax_map, ax_barcode) = plt.subplots(
    2,
    1,
    figsize=(6, 6.5),
    gridspec_kw={"height_ratios": [10, 1]},
)

anim = animate_stop_dashboard(
    data=traj.assign(
        cluster=case["labels"](
            traj,
            traj_cols=tc,
            **case["params"],
        ).to_numpy()
    ),
    stops=stops,
    traj_cols=tc,
    show_path=False,
    show_stop_overlays=False,
    ping_color="cluster",
    ping_cmap="inferno_r",
    ping_size=30,
    base_geometry=city,
    base_geom_color="#8c8c8c",
    base_geom_background="#d3d3d3",
    ax_map=ax_map,
    ax_barcode=ax_barcode,
)

plt.tight_layout(pad=0.1)
fig.subplots_adjust(top=0.98, bottom=0.12, hspace=0.12)
html = anim.to_html5_video()
plt.close(fig)
HTML(html.replace("<video ", "<video autoplay muted playsinline ", 1))